# 面试问题：LLM Scaling Law 是什么？给定训练算力、数据和推理量，怎样选择参数量与 token 数？

**一句话回答**：Scaling Law 是在限定数据分布、模型族和训练配方下拟合出的经验损失曲线，不是“参数越大越好”的定律。先用近似训练计算量 `C≈6ND` 建立候选，再用验证损失、数据上限、硬件利用率和全生命周期推理成本筛选；最后必须通过小规模 pilot 拟合不确定区间。

本 Notebook 从零实现损失曲线、解析最优解、离散预算搜索、数据重复惩罚、推理感知选型和集群工期估算。受控系数只用于展示方法，不能拿来预测真实模型。


In [ ]:
from dataclasses import dataclass
import math
import numpy as np

SEED127=12701; rng127=np.random.default_rng(SEED127)
assert SEED127==12701
assert math.isclose(6*1e9*20e9,1.2e20)
assert np.isfinite(rng127.normal())


## 1. 先冻结单位与实验边界

`N` 是参与训练的非 embedding/总参数口径之一，`D` 是实际消费 token，`C` 是训练 FLOPs。不同论文的计数口径、tokenizer、数据质量和架构不同，系数不能直接横向搬运。面试时先声明采用 `6ND` 近似，并区分“理论 FLOPs”和“集群实际吞吐”。


In [ ]:
@dataclass(frozen=True)
class TrainSpec127:
    params:float; tokens:float; non_embed:bool=True
    def __post_init__(self):
        if self.params<=0 or self.tokens<=0: raise ValueError("positive_N_D_required")
    @property
    def flops(self): return 6.0*self.params*self.tokens
tiny127=TrainSpec127(1e9,20e9)
assert tiny127.flops==1.2e20
assert tiny127.non_embed
try: TrainSpec127(0,1); raise AssertionError("invalid accepted")
except ValueError as e: assert str(e)=="positive_N_D_required"


## 2. 拟合的是可约损失，不是万能精度公式

常见形式为 `L(N,D)=E+A/N^α+B/D^β`：`E` 是不可约项，另外两项描述模型受限和数据受限区域。它假设训练配方和数据分布近似不变。真实工作应以多个规模 pilot 的 held-out loss 拟合，并检查残差，而不是把公开系数当业务真值。


In [ ]:
def loss127(N,D,E=1.5,A=120.0,B=90.0,alpha=.34,beta=.28):
    if N<=0 or D<=0: raise ValueError("domain")
    return E+A*N**(-alpha)+B*D**(-beta)
l_small127=loss127(1e8,2e9); l_big127=loss127(1e9,2e10)
assert l_big127<l_small127
assert loss127(1e9,4e10)<loss127(1e9,2e10)
assert loss127(2e9,2e10)<loss127(1e9,2e10)


## 3. 在固定计算量下求解析最优点

代入 `D=C/(6N)` 后对 `N` 求导，可得到参数量与数据量的连续最优值。指数决定预算增加时两者如何缩放；这比死记“每参数多少 token”更可靠。解析点只是候选中心，最终还要满足显存、并行度、数据许可证和最小 batch 等离散约束。


In [ ]:
def analytic_opt127(C,A=120.0,B=90.0,alpha=.34,beta=.28):
    N=((alpha*A)/(beta*B)*(C/6.0)**beta)**(1.0/(alpha+beta))
    D=C/(6.0*N)
    return N,D
N127,D127=analytic_opt127(1e21)
assert math.isclose(6*N127*D127,1e21,rel_tol=1e-12)
assert N127>0 and D127>0
assert loss127(N127,D127)<loss127(N127/4,D127*4)


## 4. 工程上用离散搜索承接硬约束

芯片数、张量并行整除、词表和层数使模型规格离散化。对合法候选计算可训练 token、预测损失和工期，再保留 Pareto frontier。这里用显式循环搜索，不调用黑盒优化器，便于解释每个候选为什么被淘汰。


In [ ]:
def grid_opt127(C,param_grid):
    rows=[]
    for N in param_grid:
        D=C/(6*N); rows.append({"N":N,"D":D,"loss":loss127(N,D)})
    return min(rows,key=lambda x:x["loss"]),rows
grid127=np.geomspace(2e8,8e9,25); best127,rows127=grid_opt127(1e21,grid127)
assert best127["loss"]==min(r["loss"] for r in rows127)
assert math.isclose(6*best127["N"]*best127["D"],1e21,rel_tol=1e-12)
assert min(grid127)<=best127["N"]<=max(grid127)


## 5. 数据不是可以无限重复的同质 token

唯一高质量 token 有上限；重复 epoch 会改变样本相关性，新增低质量数据也可能降低收益。容量规划应记录 source、license、去重版本和 mixture。教学模型在超过唯一 token 上限后增加重复惩罚，用来证明“满足 FLOPs”不等于“获得等价有效数据”。


In [ ]:
def effective_loss127(N,D,unique_cap):
    base=loss127(N,min(D,unique_cap))
    repeats=max(0.0,D/unique_cap-1.0)
    return base+.03*repeats**1.3
cap127=30e9
assert math.isclose(effective_loss127(1e9,cap127,cap127),loss127(1e9,cap127))
assert effective_loss127(1e9,120e9,cap127)>effective_loss127(1e9,60e9,cap127)
assert effective_loss127(2e9,20e9,cap127)<effective_loss127(1e9,20e9,cap127)


## 6. 训练最优不等于产品全生命周期最优

若模型要服务海量请求，较小模型多训练一些 token，可能以相近质量显著降低长期推理 FLOPs、显存和副本数。应在达到质量门槛的候选中比较 `train_cost + request_count × output_tokens × decode_cost`，并把延迟 SLO 与峰值容量作为硬约束。


In [ ]:
def lifecycle127(N,D,served_tokens,train_weight=1.0):
    return train_weight*6*N*D + 2*N*served_tokens
candidates127=[(1e9,45e9),(2e9,20e9),(4e9,10e9)]
feasible127=[x for x in candidates127 if loss127(*x)<1.705]
costs127=[lifecycle127(N,D,2e12) for N,D in feasible127]
assert len(feasible127)>=2
assert all(c>0 for c in costs127)
assert feasible127[int(np.argmin(costs127))][0]<=max(n for n,_ in feasible127)


## 7. 系数不确定性必须进入决策

pilot 数量有限、训练噪声和下游指标滞后都会让最优点漂移。可对拟合系数或实验样本 bootstrap，报告参数量/数据量区间；如果两个候选区间高度重叠，应优先选择部署更简单、可回滚且数据风险更低的方案，而不是宣称小数点级最优。


In [ ]:
samples127=[]
for _ in range(200):
    a=max(.1,rng127.normal(.34,.02)); b=max(.1,rng127.normal(.28,.02))
    samples127.append(analytic_opt127(1e21,alpha=a,beta=b)[0])
q127=np.quantile(samples127,[.1,.5,.9])
assert q127[0]<q127[1]<q127[2]
assert all(np.isfinite(q127))
assert q127[2]/q127[0]<10


## 8. 把 FLOPs 转成可审计的集群工期

理论峰值必须乘以 model FLOPs utilization；还要扣除故障、checkpoint、数据等待和评测。容量表应输出假设，而不是只给“需要多少张卡”。若预计工期超过数据/产品窗口，应回到候选集缩小模型、减少 token 或提高并行效率。


In [ ]:
def train_days127(spec,gpus,peak_tflops,mfu,availability=.95):
    useful=gpus*peak_tflops*1e12*mfu*availability
    return spec.flops/useful/86400
chosen127=TrainSpec127(best127["N"],best127["D"]); days127=train_days127(chosen127,64,312,.42)
assert days127>0
assert train_days127(chosen127,128,312,.42)<days127
assert train_days127(chosen127,64,312,.2)>days127


## 面试总结

完整回答是：**统一 `N/D/C` 口径 → pilot 拟合损失曲线与残差 → `6ND` 下求连续候选 → 离散架构/硬件搜索 → 加数据上限和质量 → 加推理全生命周期成本 → bootstrap 不确定区间 → 用 MFU、可用率换算工期并做实际小规模验证**。Scaling Law 是决策仪表，不是脱离训练配方的自然常数。

延伸阅读：[Chinchilla](https://arxiv.org/abs/2203.15556)、[Scaling Laws for Neural Language Models](https://arxiv.org/abs/2001.08361)、[Beyond Chinchilla-Optimal](https://arxiv.org/abs/2401.00448)。
